# Kinship Library
## Notebook 01: Collection, Data Quality and Exploratory Analysis

### Goal

Build a trustworthy thematic book catalogue before modeling.

This notebook covers the useful work from the original Day 1 and Day 2.

### Main project question

Can book descriptions and subject metadata help readers discover meaningful books about living in connection with self, community, nature, and ancestral knowledge?

### Data principles

We keep source metadata separate from our later reader paths.

We do not assume that Yoga, Ayurveda, Indian spirituality, Kogi knowledge, Muisca knowledge, and ecological philosophy are interchangeable.

We keep the source and search query for every record so that the collection process remains traceable.

### Outputs

`data/raw/open_library_books.csv`

`data/raw/books_to_scrape.csv`

`data/processed/books_clean.csv`

## 1. Setup

### Purpose

Import simple libraries and define repository relative paths.

### Expected result

The folders are created automatically. The same notebook can run in Colab or from a cloned repository.

In [12]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/kinship_library"
)

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"

RAW_DATA.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_DATA.mkdir(
    parents=True,
    exist_ok=True
)

MODELS.mkdir(
    parents=True,
    exist_ok=True
)

RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA)
print("Processed data:", PROCESSED_DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/kinship_library
Raw data: /content/drive/MyDrive/kinship_library/data/raw
Processed data: /content/drive/MyDrive/kinship_library/data/processed


## 2. Business understanding

### Reader need

Kinship Library is designed for readers who want to explore questions of ecological connection, meaning, embodied practice, Yoga philosophy, Ayurveda, Indian spirituality, and selected Indigenous knowledge traditions from South America.

Later, a reader may describe an experience such as fear of the future, ecological concern, disconnection from nature, or a search for meaning.

The recommender will not diagnose that person. It will retrieve books whose themes are semantically related to the reader's self described interest.

### Questions for this phase

1. Can we collect enough relevant books with meaningful descriptions and subjects?
2. Which sources provide the most complete metadata?
3. Which themes are overrepresented or underrepresented?
4. How much missing information do we have?
5. Are there extreme values or suspicious records that need inspection?

## 3. Search vocabulary

These queries are collection tools, not labels about what a book can psychologically help with.

In [13]:
THEME_QUERIES = {

    "humans_and_nature": [
        "ecology human nature",
        "environmental humanities",
        "more than human",
        "ecofeminism",
        "deep ecology",
        "human nature connection"
    ],

    "yoga_philosophy": [
        "Yoga Sutras Patanjali",
        "Vedas",
        "Upanishads",
        "Bhagavad Gita",
        "yoga philosophy"
    ],

    "yoga_way_of_life": [
        "eight limbs yoga",
        "pranayama yoga",
        "meditation yoga",
        "yoga way of life"
    ],

    "ayurveda_indian_spirituality": [
        "Ayurveda",
        "Vedanta",
        "Indian spirituality",
        "Autobiography of a Yogi",
        "Indian Tantra"
    ],

    "indigenous_knowledge_americas": [

        # Broad Indigenous knowledge
        "Traditional Ecological Knowledge",
        "Indigenous Knowledge ecology",
        "Indigenous ecological knowledge",
        "Indigenous land based knowledge",
        "Indigenous environmental knowledge",
        "biocultural knowledge Indigenous",

        # Cosmology and spirituality
        "Indigenous cosmology Americas",
        "Indigenous worldviews nature",
        "Indigenous spirituality nature",
        "Indigenous peoples human nature relationship",

        # South America
        "South American Indigenous knowledge",
        "South American Indigenous cosmology",
        "Andean Indigenous knowledge",
        "Andean cosmology",
        "Amazonian Indigenous knowledge",
        "Amazonian cosmology",
        "Amerindian cosmology",
        "Amerindian perspectivism",

        # Colombia
        "Indigenous knowledge Colombia",
        "Indigenous cosmology Colombia",
        "Kogi Colombia",
        "Kogi worldview",
        "Kogi cosmology",
        "Muisca Colombia",
        "Muisca worldview",
        "Muisca cosmology",
        "Kamentsa Colombia",
        "Embera Colombia",
        "Nasa Colombia",
        "Wayuu Colombia",

        # Other traditions of the Americas
        "Quechua Indigenous knowledge",
        "Aymara Indigenous knowledge",
        "Mapuche Indigenous knowledge",
        "Maya Indigenous cosmology",
        "Native American ecological knowledge"
    ]
}


SEED_TITLES = [
    "Staying with the Trouble",
    "Ancestral Future",
    "The Yoga Sutras of Patanjali",
    "Autobiography of a Yogi",
    "Bhagavad Gita",
]

## 4. Open Library collection

### Purpose

Use the Open Library Search API to collect multiple books per request.

### Important

The code caches the result as CSV so we do not call the API again every time we rerun the notebook.

In [14]:
OPEN_LIBRARY_FILE = RAW_DATA / "open_library_books.csv"

if OPEN_LIBRARY_FILE.exists():
    open_library_books = pd.read_csv(OPEN_LIBRARY_FILE)
    print("Loaded cached Open Library data:", len(open_library_books))

else:
    rows = []

    headers = {
        "User-Agent": "KinshipLibraryStudentProject"
    }

    for collection_theme, queries in THEME_QUERIES.items():

        print("\nCollecting theme:", collection_theme)

        for query in queries:

            print("Query:", query)

            params = {
                "q": query,
                "limit": 30,
                "fields": (
                    "key,title,author_name,first_publish_year,"
                    "subject,isbn,language,ratings_average,ratings_count"
                )
            }

            try:
                response = requests.get(
                    "https://openlibrary.org/search.json",
                    params=params,
                    headers=headers,
                    timeout=60
                )

                if response.status_code != 200:
                    print(
                        "Skipped:",
                        query,
                        "Status:",
                        response.status_code
                    )
                    continue

                data = response.json()

                for book in data.get("docs", []):

                    rows.append({
                        "source_id": book.get("key"),
                        "title": book.get("title"),
                        "author": ", ".join(
                            book.get("author_name", [])
                        ),
                        "description": np.nan,
                        "subjects": ", ".join(
                            book.get("subject", [])[:15]
                        ),
                        "published_year": book.get(
                            "first_publish_year"
                        ),
                        "average_rating": book.get(
                            "ratings_average"
                        ),
                        "ratings_count": book.get(
                            "ratings_count"
                        ),
                        "isbn": (
                            book.get("isbn", [np.nan])[0]
                            if book.get("isbn")
                            else np.nan
                        ),
                        "language": (
                            book.get("language", [np.nan])[0]
                            if book.get("language")
                            else np.nan
                        ),
                        "source": "open_library",
                        "collection_theme": collection_theme,
                        "search_query": query
                    })

                print(
                    "Collected:",
                    len(data.get("docs", [])),
                    "books"
                )

                # Give Open Library some time
                time.sleep(2)

            except requests.exceptions.Timeout:

                print(
                    "Timeout for:",
                    query,
                    "Skipping this query and continuing."
                )

                time.sleep(5)
                continue

            except requests.exceptions.RequestException as error:

                print(
                    "Request error for:",
                    query
                )

                print(error)

                time.sleep(5)
                continue

    open_library_books = pd.DataFrame(rows)

    open_library_books.to_csv(
        OPEN_LIBRARY_FILE,
        index=False
    )

    print("\nCollection finished")
    print("Total books collected:", len(open_library_books))
    print("Saved to:", OPEN_LIBRARY_FILE)

open_library_books.head()


Query: ecology human nature
Collected: 30 books
Query: environmental humanities
Collected: 30 books
Query: more than human
Collected: 30 books
Query: ecofeminism
Collected: 30 books
Query: deep ecology
Collected: 30 books
Query: human nature connection
Collected: 30 books

Query: Yoga Sutras Patanjali
Collected: 30 books
Query: Vedas
Collected: 30 books
Query: Upanishads
Collected: 30 books
Query: Bhagavad Gita
Collected: 30 books
Query: yoga philosophy
Collected: 30 books

Query: eight limbs yoga
Collected: 21 books
Query: pranayama yoga
Collected: 30 books
Query: meditation yoga
Collected: 30 books
Query: yoga way of life
Collected: 30 books

Query: Ayurveda
Collected: 30 books
Query: Vedanta
Collected: 30 books
Query: Indian spirituality
Collected: 30 books
Query: Autobiography of a Yogi
Collected: 30 books
Query: Indian Tantra
Collected: 30 books

Query: Traditional Ecological Knowledge
Collected: 30 books
Query: Indigenous Knowledge ecology
Collected: 30 books
Query: Indigenous e

,source_id,title,author,description,subjects,published_year,average_rating,ratings_count,isbn,language,source,collection_theme,search_query
0,/works/OL6772557W,The ecology of human development,Urie Bronfenbrenner,NaN,"Child psychology, Research, Human Development,...",1979.0,5.000000,1.0,9780674224575,eng,open_library,humans_and_nature,ecology human nature
1,/works/OL3458484W,People and nature,Emilio F. Moran,NaN,"Effect of environment on, Effect of environmen...",2006.0,NaN,NaN,1118877411,eng,open_library,humans_and_nature,ecology human nature
2,/works/OL2990533W,The green imperative,"Victor J. Papanek, Victor Papanek",NaN,"Design, industrial, Environmental aspects, Env...",1995.0,NaN,NaN,9780500296196,eng,open_library,humans_and_nature,ecology human nature
3,/works/OL1805267W,Design with nature,Ian L. McHarg,NaN,"Effect of environment on, Effect of human bein...",1969.0,4.333334,6.0,047111460X,eng,open_library,humans_and_nature,ecology human nature
4,/works/OL2911248W,"The death of nature: women, ecology, and the s...",Carolyn Merchant,NaN,"Human ecology, Philosophy of nature, Social as...",1980.0,NaN,NaN,9780062505712,eng,open_library,humans_and_nature,ecology human nature


In [15]:
print("Books collected:", len(open_library_books))
print("Saved at:", OPEN_LIBRARY_FILE)
print("File exists:", OPEN_LIBRARY_FILE.exists())

Books collected: 1075
Saved at: /content/drive/MyDrive/kinship_library/data/raw/open_library_books.csv
File exists: True


In [16]:
print(OPEN_LIBRARY_FILE)

print(
    "File exists:",
    OPEN_LIBRARY_FILE.exists()
)

/content/drive/MyDrive/kinship_library/data/raw/open_library_books.csv
File exists: True


## 5. Books to Scrape sample

### Purpose

Demonstrate web scraping with a site built for scraping practice.

This sample is kept separate from the thematic model because it does not provide the rich descriptions we need for semantic recommendation.

In [ ]:
SCRAPED_FILE = RAW_DATA / "books_to_scrape.csv"

if SCRAPED_FILE.exists():
    scraped_books = pd.read_csv(SCRAPED_FILE)
    print("Loaded cached scraped data:", len(scraped_books))

else:
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    rows = []

    for page in range(1, 6):
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"
        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            print("Skipped page:", page)
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        for book in soup.select("article.product_pod"):
            rating_word = book.p.get("class", ["", ""])[1]

            rows.append({
                "title": book.h3.a["title"],
                "price": book.select_one(".price_color").get_text(strip=True),
                "rating": rating_map.get(rating_word),
                "availability": book.select_one(".availability").get_text(strip=True),
                "source": "books_to_scrape"
            })

        time.sleep(1)

    scraped_books = pd.DataFrame(rows)
    scraped_books.to_csv(SCRAPED_FILE, index=False)

scraped_books.head()

,title,price,rating,availability,source
0,A Light in the Attic,Â£51.77,3,In stock,books_to_scrape
1,Tipping the Velvet,Â£53.74,1,In stock,books_to_scrape
2,Soumission,Â£50.10,1,In stock,books_to_scrape
3,Sharp Objects,Â£47.82,4,In stock,books_to_scrape
4,Sapiens: A Brief History of Humankind,Â£54.23,5,In stock,books_to_scrape


## 7. Clean the thematic catalogue

### Purpose

Standardize types, remove obvious duplicates, inspect missing values, and create simple quality features.

In [17]:
books = open_library_books.copy()

books["title"] = books["title"].astype("string").str.strip()
books["author"] = books["author"].astype("string").str.strip()
books["subjects"] = books["subjects"].astype("string").str.strip()

books["published_year"] = pd.to_numeric(
    books["published_year"],
    errors="coerce"
)

books["average_rating"] = pd.to_numeric(
    books["average_rating"],
    errors="coerce"
)

books["ratings_count"] = pd.to_numeric(
    books["ratings_count"],
    errors="coerce"
)

books = books.dropna(subset=["title"])

books = books.drop_duplicates(
    subset=["title", "author"],
    keep="first"
)

books = books.reset_index(drop=True)

books["subject_count"] = (
    books["subjects"]
    .fillna("")
    .apply(lambda text: 0 if text == "" else len(text.split(",")))
)

books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   source_id         925 non-null    object 
 1   title             925 non-null    string 
 2   author            925 non-null    string 
 3   description       0 non-null      float64
 4   subjects          925 non-null    string 
 5   published_year    917 non-null    float64
 6   average_rating    113 non-null    float64
 7   ratings_count     113 non-null    float64
 8   isbn              802 non-null    object 
 9   language          878 non-null    object 
 10  source            925 non-null    object 
 11  collection_theme  925 non-null    object 
 12  search_query      925 non-null    object 
 13  subject_count     925 non-null    int64  
dtypes: float64(4), int64(1), object(6), string(3)
memory usage: 101.3+ KB


In [18]:
CLEAN_FILE = PROCESSED_DATA / "books_clean.csv"

books.to_csv(
    CLEAN_FILE,
    index=False
)

print("Saved:", CLEAN_FILE)
print("Rows:", len(books))
print("File exists:", CLEAN_FILE.exists())

Saved: /content/drive/MyDrive/kinship_library/data/processed/books_clean.csv
Rows: 925
File exists: True


## 7.1 Looking for descriptions

In [19]:
import requests
import time


def get_open_library_description(source_id):

    if pd.isna(source_id):
        return np.nan

    work_id = str(source_id).replace("/works/", "")

    url = (
        "https://openlibrary.org/works/"
        + work_id
        + ".json"
    )

    try:

        response = requests.get(
            url,
            timeout=30
        )

        if response.status_code != 200:
            return np.nan

        data = response.json()

        description = data.get(
            "description"
        )

        if isinstance(description, dict):
            description = description.get(
                "value"
            )

        if description:
            return description

        return np.nan

    except requests.exceptions.RequestException:
        return np.nan

## Missing values

In [20]:
missing_values = pd.DataFrame({
    "missing_count": books.isnull().sum(),
    "missing_percent": (books.isnull().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

missing_values

,missing_count,missing_percent
description,925,100.00
average_rating,812,87.78
ratings_count,812,87.78
isbn,123,13.30
language,47,5.08
published_year,8,0.86
subjects,0,0.00
author,0,0.00
source_id,0,0.00
title,0,0.00


In [21]:
print("Total collected records:", len(open_library_books))

print(
    "Unique Open Library IDs:",
    open_library_books["source_id"].nunique()
)

duplicate_ids = (
    open_library_books["source_id"]
    .duplicated(keep=False)
    .sum()
)

print(
    "Records belonging to duplicated book IDs:",
    duplicate_ids
)

Total collected records: 1075
Unique Open Library IDs: 962
Records belonging to duplicated book IDs: 184


In [22]:
print("Total books:", len(open_library_books))

display(
    open_library_books[
        "collection_theme"
    ].value_counts()
)

display(
    open_library_books[
        [
            "title",
            "author",
            "collection_theme",
            "search_query"
        ]
    ].sample(
        min(20, len(open_library_books)),
        random_state=42
    )
)

Total books: 1075


,count
collection_theme,
indigenous_knowledge_americas,484
humans_and_nature,180
yoga_philosophy,150
ayurveda_indian_spirituality,150
yoga_way_of_life,111


,title,author,collection_theme,search_query
428,The Way of Aikido,George Burr Leonard,yoga_way_of_life,yoga way of life
312,I-God,Gnanavallal Paranjothi Mahan,yoga_philosophy,yoga philosophy
306,Tree of Yoga,B. K. S. Iyengar,yoga_philosophy,yoga philosophy
141,Deep ecology,Michael Tobias,humans_and_nature,deep ecology
273,The Bhagavad-Gita,Edwin Arnold,yoga_philosophy,Bhagavad Gita
996,El pensamiento de la comunidad nasa yuwe,Alexander Ávila Martínez,indigenous_knowledge_americas,Nasa Colombia
825,The diversity of international law,Kalliopi Koufa,indigenous_knowledge_americas,Indigenous peoples human nature relationship
247,Twenty-Eight Upanishads (Îsha & Others).,Upanishads Upanishads,yoga_philosophy,Upanishads
583,The Niśvāsatattvasaṃhitā,"Dominic Goodall, Alexis Sanderson, H. Isaacson",ayurveda_indian_spirituality,Indian Tantra
51,Environmental science,"Eldon D. Enger, Richard J. Kormelink, Bradley ...",humans_and_nature,environmental humanities


In [25]:
duplicate_books = (
    open_library_books[
        open_library_books["source_id"].duplicated(
            keep=False
        )
    ]
    .sort_values("source_id")
)

display(
    duplicate_books[
        [
            "source_id",
            "title",
            "author",
            "collection_theme",
            "search_query"
        ]
    ].head()
)

,source_id,title,author,collection_theme,search_query
627,/works/OL1054919W,Biopiracy,Vandana Shiva,indigenous_knowledge_americas,Indigenous Knowledge ecology
701,/works/OL1054919W,Biopiracy,Vandana Shiva,indigenous_knowledge_americas,Indigenous environmental knowledge
657,/works/OL1054919W,Biopiracy,Vandana Shiva,indigenous_knowledge_americas,Indigenous ecological knowledge
218,/works/OL11500W,The religion and philosophy of the Veda and Up...,Arthur Berriedale Keith,yoga_philosophy,Vedas
246,/works/OL11500W,The religion and philosophy of the Veda and Up...,Arthur Berriedale Keith,yoga_philosophy,Upanishads


In [26]:
indigenous_books = open_library_books[
    open_library_books["collection_theme"]
    == "indigenous_knowledge_americas"
]

display(
    indigenous_books[
        [
            "title",
            "author",
            "search_query",
            "subjects"
        ]
    ].sample(
        min(30, len(indigenous_books)),
        random_state=42
    )
)

,title,author,search_query,subjects
1009,Study of future world markets for agricultural...,F. W. Gobetz,Nasa Colombia,"Aeronautics in agriculture, Aerial fertilizing"
1065,The Water Walker,Joanne Robertson,Native American ecological knowledge,"JUVENILE NONFICTION, Savoirs écologiques tradi..."
772,Thai folklore,"Sirāphō̜n Thitathān Na Thalāng, Čhulālongkō̜nm...",Indigenous worldviews nature,"Social life and customs, Folklore, Tales, Mann..."
1037,Desecrated land,"Roberto Pineda Camacho, Organización Nacional ...",Wayuu Colombia,"Natural resources, Economic development projec..."
888,The occult life of things,Fernando Santos-Granero,Amazonian cosmology,"Material culture, Indians of South America, Ce..."
739,Biodiversity and the law,Charles R. McManis,biocultural knowledge Indigenous,"Intellectual property, Biotechnology, Law and ..."
971,Semejantes a los dioses,Luis Guillermo Vasco Uribe,Embera Colombia,"Chamí baskets, Chamí pottery, Keramik, Korbwar..."
977,El jaibaná en la encrucijada,Pablo Jaramillo Salazar,Embera Colombia,"Embera Indians, Rites and ceremonies, Religion..."
1070,Understanding child abuse and neglect,Cynthia Crosson-Tower,Native American ecological knowledge,"Social work with children, Services for, Child..."
818,The Oxford handbook on the United Nations,"Thomas George Weiss, Sam Daws",Indigenous peoples human nature relationship,"Handbooks, manuals, United Nations, Internatio..."


## 8. Google Books Enrichment

Our first Colab test returned HTTP 429 even for one request.

But, I got an API Books, and created a project and now we can enrich the descriptions with google books




```
# Open Library clean catalogue
          ↓
       Has ISBN?
       ↙      ↘
     Yes       No
      ↓         ↓
ISBN search   Title + author search
      ↓         ↓
       Google Books
            ↓
Verify match
            ↓
Collect description
categories
publisher
Google Books ID
            ↓
Cache results
            ↓
Merge with books
```



In [27]:
import requests
import time
import numpy as np
import pandas as pd

In [28]:
from google.colab import userdata

GOOGLE_BOOKS_API_KEY = userdata.get(
    "API_key_books"
)

print("API key loaded successfully")

API key loaded successfully


## 8.2 Full Google Books enrichment cell

In [29]:
from google.colab import userdata
import requests
import time
import pandas as pd
import numpy as np

GOOGLE_BOOKS_API_KEY = userdata.get(
    "API_key_books"
)

GOOGLE_CACHE_FILE = (
    RAW_DATA / "google_books_enrichment.csv"
)

print("Cache location:")
print(GOOGLE_CACHE_FILE)

Cache location:
/content/drive/MyDrive/kinship_library/data/raw/google_books_enrichment.csv


In [30]:
def search_google_book(row):

    url = "https://www.googleapis.com/books/v1/volumes"

    isbn = row["isbn"]
    title = row["title"]
    author = row["author"]

    # Prefer ISBN because it gives us a stronger match
    if pd.notna(isbn):
        query = f"isbn:{isbn}"

    else:
        query = f'intitle:"{title}"'

        if pd.notna(author):
            query += f' inauthor:"{author}"'

    params = {
        "q": query,
        "maxResults": 1,
        "printType": "books",
        "key": GOOGLE_BOOKS_API_KEY
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        # Successful request
        if response.status_code == 200:

            data = response.json()

            # Google searched successfully
            # but did not find the book
            if data.get("totalItems", 0) == 0:

                return {
                    "status": "not_found"
                }

            item = data["items"][0]

            info = item.get(
                "volumeInfo",
                {}
            )

            return {
                "status": "found",
                "google_books_id": item.get("id"),
                "google_title": info.get("title"),
                "google_authors": ", ".join(
                    info.get("authors", [])
                ),
                "description": info.get("description"),
                "categories": ", ".join(
                    info.get("categories", [])
                ),
                "publisher": info.get("publisher"),
                "google_published_date": info.get(
                    "publishedDate"
                ),
                "google_average_rating": info.get(
                    "averageRating"
                ),
                "google_ratings_count": info.get(
                    "ratingsCount"
                )
            }

        # Rate limit
        elif response.status_code == 429:

            return {
                "status": "rate_limited"
            }

        # Other HTTP problem
        else:

            return {
                "status": (
                    "http_"
                    + str(response.status_code)
                )
            }

    except requests.exceptions.RequestException:

        return {
            "status": "request_error"
        }

In [31]:
# Load previous progress if available

if GOOGLE_CACHE_FILE.exists():

    google_cache = pd.read_csv(
        GOOGLE_CACHE_FILE
    )

    print(
        "Previous results loaded:",
        len(google_cache)
    )

else:

    google_cache = pd.DataFrame()

    print(
        "No previous cache found."
    )


# IDs already properly processed

if len(google_cache) > 0:

    completed_ids = set(
        google_cache.loc[
            google_cache["status"].isin(
                ["found", "not_found"]
            ),
            "source_id"
        ]
        .astype(str)
    )

else:

    completed_ids = set()


print(
    "Already completed:",
    len(completed_ids)
)

print(
    "Books remaining:",
    len(books) - len(completed_ids)
)

No previous cache found.
Already completed: 0
Books remaining: 925


In [32]:
new_results = []

rate_limit_reached = False


for index, row in books.iterrows():

    source_id = str(
        row["source_id"]
    )

    # Do not request books already processed
    if source_id in completed_ids:
        continue


    result = search_google_book(row)

    status = result["status"]


    # STOP if Google rate limits us
    if status == "rate_limited":

        print()
        print("Google Books rate limit reached.")
        print("Collection stopped safely.")
        print("Last attempted book:")
        print(row["title"])

        rate_limit_reached = True

        break


    # Store identifying information
    result["source_id"] = source_id
    result["open_library_title"] = row["title"]
    result["open_library_author"] = row["author"]
    result["isbn"] = row["isbn"]

    new_results.append(
        result
    )


    print(
        len(completed_ids)
        + len(new_results),
        "/",
        len(books),
        row["title"],
        "|",
        status
    )


    # Save every 10 books
    if len(new_results) >= 10:

        batch = pd.DataFrame(
            new_results
        )

        google_cache = pd.concat(
            [
                google_cache,
                batch
            ],
            ignore_index=True
        )

        google_cache.to_csv(
            GOOGLE_CACHE_FILE,
            index=False
        )

        completed_ids.update(
            batch["source_id"]
            .astype(str)
        )

        new_results = []

        print(
            "Progress saved to Drive:",
            len(google_cache)
        )


    # Slower than yesterday
    time.sleep(5)

1 / 925 The ecology of human development | found
2 / 925 People and nature | http_503
3 / 925 The green imperative | found
4 / 925 Design with nature | found
5 / 925 The death of nature: women, ecology, and the scientific revolution | found
6 / 925 Human geography | found
7 / 925 Galen's Prophecy | found
8 / 925 Man's responsibility for nature | found
9 / 925 Ecological Transition | found
10 / 925 Biomimicry | found
Progress saved to Drive: 10
11 / 925 Sex, ecology, spirituality | found
12 / 925 Team human | found
13 / 925 Ecology and equity | found
14 / 925 Human Ecology | found
15 / 925 Understanding Human Ecology | not_found
16 / 925 Ecological revolutions | not_found
17 / 925 Human ecology | found
18 / 925 Ecology | found
19 / 925 Man, nature, and ecology | found
20 / 925 Human ecology | found
Progress saved to Drive: 20
21 / 925 Environmental philosophy | found
22 / 925 Sacred ecology | not_found
23 / 925 Human dimensions of ecological restoration | found
24 / 925 The forest and t

/tmp/ipykernel_955/338118846.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  google_cache = pd.concat(


551 / 925 The Niśvāsatattvasaṃhitā | not_found
552 / 925 The world of Tantra | not_found
553 / 925 Geschichte der indischen litteratur | found
554 / 925 Buddhist Tantra | not_found
555 / 925 The Tantras and their impact on Indian life | found
556 / 925 An Ancient Indian system of rasayana | found
557 / 925 Sanskrit manuscripts from Tibet | found
558 / 925 Traditional Ecological Knowledge and Natural Resource Management | found
559 / 925 Traditional Ecological Knowledge | found
560 / 925 Traditional Ecological Knowledge in Georgia | not_found
Progress saved to Drive: 560
561 / 925 Traditional Ecological Knowledge | found
562 / 925 Traditional ecological knowledge | found
563 / 925 Traditional Ecological Knowledge and Global Pandemics | found
564 / 925 The Traditional Ecological Knowledge of the Solega | found
565 / 925 Traditional Ecological Knowledge , Ethnoecology, It's Use in Indigenous Communities and Native America | not_found
566 / 925 Traditional Ecological Knowledge | found
567 

/tmp/ipykernel_955/338118846.py:65: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  google_cache = pd.concat(


871 / 925 Kiwe Ten'za | not_found
872 / 925 Serie Rio de letras | found
873 / 925 Utopías interculturales | found
874 / 925 Construcción de micromundos | found
875 / 925 Los "teatros" de la memoria | found
876 / 925 Reconciliaciones y resistencias | not_found
877 / 925 El arco, el cuerpo y la seña | found
878 / 925 U' sxakwe sa'tkwe ena's umnxi | found
879 / 925 Experiencias de revitalización cultural y lingüística | found
880 / 925 De los saberes de la emancipación y de la dominación | found
Progress saved to Drive: 880
881 / 925 Pos-acuerdo y territorio en las comunidades indígenas, afro y campesinas en el norte del Cauca | found
882 / 925 Study of future world markets for agricultural aircraft | found
883 / 925 Mujeres indígenas haciendo, investigando y reescribiendo lo político en América Latina | found
884 / 925 Nosotros los Wayuu | found
885 / 925 Lo que saben los wayuu | found
886 / 925 Apalaanchi | found
887 / 925 El sabedor Wayuu | not_found
888 / 925 Censo binacional de la ét

In [1]:
google_test = pd.DataFrame(
    google_results
)

display(
    google_test[
        [
            "google_title",
            "google_authors",
            "description",
            "categories"
        ]
    ]
)

NameError: name 'pd' is not defined

In [ ]:
print(
    "Books found:",
    google_test["google_books_id"]
    .notna()
    .sum(),
    "out of",
    len(test_books)
)

print(
    "Descriptions found:",
    google_test["description"]
    .notna()
    .sum(),
    "out of",
    len(test_books)
)

Books found: 10 out of 10
Descriptions found: 8 out of 10


In [ ]:
GOOGLE_CACHE_FILE = (
    RAW_DATA / "google_books_enrichment.csv"
)

# Load previous progress if it exists
if GOOGLE_CACHE_FILE.exists():

    google_cache = pd.read_csv(
        GOOGLE_CACHE_FILE
    )

    completed_ids = set(
        google_cache["source_id"]
        .dropna()
        .astype(str)
    )

    print(
        "Previous results loaded:",
        len(google_cache)
    )

else:

    google_cache = pd.DataFrame()

    completed_ids = set()

    print("Starting new enrichment")


new_results = []

for index, row in books.iterrows():

    source_id = str(row["source_id"])

    # Skip books already processed
    if source_id in completed_ids:
        continue

    result = search_google_book(row)

    # Always save the source ID
    if result is None:

        result = {
            "source_id": source_id,
            "google_books_id": np.nan,
            "google_title": np.nan,
            "google_authors": np.nan,
            "description": np.nan,
            "categories": np.nan,
            "publisher": np.nan,
            "google_published_date": np.nan,
            "google_average_rating": np.nan,
            "google_ratings_count": np.nan
        }

    else:

        result["source_id"] = source_id

    new_results.append(result)

    print(
        f"{index + 1}/{len(books)}",
        row["title"]
    )

    # Save every 25 books
    if len(new_results) >= 25:

        batch = pd.DataFrame(
            new_results
        )

        google_cache = pd.concat(
            [google_cache, batch],
            ignore_index=True
        )

        google_cache.to_csv(
            GOOGLE_CACHE_FILE,
            index=False
        )

        new_results = []

        print(
            "Progress saved:",
            len(google_cache),
            "books"
        )

    time.sleep(1)


# Save the final incomplete batch
if new_results:

    batch = pd.DataFrame(
        new_results
    )

    google_cache = pd.concat(
        [google_cache, batch],
        ignore_index=True
    )

    google_cache.to_csv(
        GOOGLE_CACHE_FILE,
        index=False
    )


print("\nEnrichment finished")

print(
    "Books processed:",
    len(google_cache)
)

Starting new enrichment
1/925 The ecology of human development
2/925 People and nature
3/925 The green imperative
4/925 Design with nature
5/925 The death of nature: women, ecology, and the scientific revolution
6/925 Human geography
7/925 Galen's Prophecy
8/925 Man's responsibility for nature
9/925 Ecological Transition
10/925 Biomimicry
11/925 Sex, ecology, spirituality
12/925 Team human
13/925 Ecology and equity
14/925 Human Ecology
15/925 Understanding Human Ecology
16/925 Ecological revolutions
17/925 Human ecology
18/925 Ecology
19/925 Man, nature, and ecology
20/925 Human ecology
21/925 Environmental philosophy
22/925 Sacred ecology
23/925 Human dimensions of ecological restoration
24/925 The forest and the sea
25/925 Man's nature and nature's man
Progress saved: 25 books
26/925 Being Ecological
27/925 Human-nature interactions in the anthropocene
28/925 Last Child in the Woods
29/925 Islands of Abandonment
30/925 Natural capital and human economic survival
31/925 Environmental 

In [ ]:
print("Total attempted:", len(google_cache))

print(
    "Successful Google matches:",
    google_cache["google_books_id"].notna().sum()
)

print(
    "Descriptions found:",
    google_cache["description"].notna().sum()
)

print(
    "Failed or unavailable:",
    google_cache["google_books_id"].isna().sum()
)

Total attempted: 925
Successful Google matches: 80
Descriptions found: 65
Failed or unavailable: 845


## 9. Univariate exploration

In [ ]:
plt.figure(figsize=(8, 5))
books["average_rating"].dropna().hist(bins=15)
plt.title("Distribution of average ratings")
plt.xlabel("Average rating")
plt.ylabel("Number of books")
plt.show()

plt.figure(figsize=(8, 5))
books["subject_count"].hist(bins=20)
plt.title("Distribution of subject count")
plt.xlabel("Number of subjects")
plt.ylabel("Number of books")
plt.show()

In [ ]:
google_cache["status"] = np.where(
    google_cache["google_books_id"].notna(),
    "found",
    "needs_retry"
)

print(
    google_cache["status"].value_counts()
)

status
needs_retry    845
found           80
Name: count, dtype: int64


In [ ]:
google_cache.to_csv(
    GOOGLE_CACHE_FILE,
    index=False
)

In [ ]:
successful_ids = set(
    google_cache.loc[
        google_cache["status"] == "found",
        "source_id"
    ].astype(str)
)

retry_books = books[
    ~books["source_id"]
    .astype(str)
    .isin(successful_ids)
].copy()

print(
    "Books remaining:",
    len(retry_books)
)

Books remaining: 845


In [4]:
import pandas as pd
import numpy as np
import requests
import time

from pathlib import Path
from google.colab import userdata

GOOGLE_BOOKS_API_KEY = userdata.get(
    "API_key_books"
)

In [5]:
from pathlib import Path

list(Path("/content").rglob("google_books_enrichment.csv"))

[]

In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/kinship_library"
)

RAW_DATA = DRIVE_PROJECT / "data" / "raw"
PROCESSED_DATA = DRIVE_PROJECT / "data" / "processed"

RAW_DATA.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_DATA.mkdir(
    parents=True,
    exist_ok=True
)

print("Project folder:", DRIVE_PROJECT)

Project folder: /content/drive/MyDrive/kinship_library


In [8]:
list(
    Path("/content/drive/MyDrive").rglob(
        "open_library_books.csv"
    )
)

[]

## 10. Outlier check

We use the IQR rule to flag unusual values. Flagging an outlier does not mean deleting it.

In [ ]:
q1 = books["subject_count"].quantile(0.25)
q3 = books["subject_count"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

subject_outliers = books[
    (books["subject_count"] < lower_bound)
    | (books["subject_count"] > upper_bound)
]

print("LCV for IQR inspection:", lower_bound)
print("UCV for IQR inspection:", upper_bound)
print("Outliers:", len(subject_outliers))

plt.figure(figsize=(8, 4))
plt.boxplot(books["subject_count"].dropna(), vert=False)
plt.title("Subject count outlier check")
plt.xlabel("Number of subjects")
plt.show()

## 11. Probability

Estimate the probability that a randomly selected record in our collected sample has at least five subjects.

This probability describes our sample, not the entire population of books.

In [ ]:
has_rich_subject_metadata = books["subject_count"] >= 5
probability_rich_metadata = has_rich_subject_metadata.mean()

print(
    "Estimated probability:",
    round(probability_rich_metadata, 3)
)

## 12. Bivariate exploration and correlation

In [ ]:
correlation_data = books[
    ["average_rating", "ratings_count", "published_year", "subject_count"]
].copy()

correlation_matrix = correlation_data.corr(numeric_only=True)
correlation_matrix

## 13. Normal distribution check for ratings

Null hypothesis: average ratings are compatible with a normal distribution.

Alternative hypothesis: average ratings are not compatible with a normal distribution.

Alpha level: 0.05.

In [ ]:
rating_values = books["average_rating"].dropna()

if len(rating_values) >= 3:
    sample = rating_values.sample(
        min(len(rating_values), 500),
        random_state=RANDOM_STATE
    )

    statistic, p_value = stats.shapiro(sample)

    print("Test statistic:", round(statistic, 4))
    print("P value:", round(p_value, 4))
    print("Alpha:", 0.05)

    if p_value < 0.05:
        print("Decision: reject the null hypothesis")
    else:
        print("Decision: fail to reject the null hypothesis")
else:
    print("Not enough rating observations")

## 14. Bivariate hypothesis test with rejection regions

Question: Is mean subject count different between the humans and nature collection and Yoga philosophy collection?

Null hypothesis: the two population means are equal.

Alternative hypothesis: the two population means are different.

Test: two sided Welch t test.

Alpha: 0.05.

In [ ]:
group_a = books.loc[
    books["collection_theme"] == "humans_and_nature",
    "subject_count"
].dropna()

group_b = books.loc[
    books["collection_theme"] == "yoga_philosophy",
    "subject_count"
].dropna()

alpha = 0.05

if len(group_a) >= 2 and len(group_b) >= 2:

    test_statistic, p_value = stats.ttest_ind(
        group_a,
        group_b,
        equal_var=False
    )

    var_a = group_a.var(ddof=1)
    var_b = group_b.var(ddof=1)
    n_a = len(group_a)
    n_b = len(group_b)

    df_num = (var_a / n_a + var_b / n_b) ** 2
    df_den = (
        ((var_a / n_a) ** 2) / (n_a - 1)
        + ((var_b / n_b) ** 2) / (n_b - 1)
    )
    degrees_freedom = df_num / df_den

    lcv = stats.t.ppf(alpha / 2, degrees_freedom)
    ucv = stats.t.ppf(1 - alpha / 2, degrees_freedom)

    print("Test statistic:", round(test_statistic, 4))
    print("P value:", round(p_value, 4))
    print("Alpha:", alpha)
    print("LCV:", round(lcv, 4))
    print("UCV:", round(ucv, 4))

    if p_value < alpha:
        print("Decision: reject the null hypothesis")
    else:
        print("Decision: fail to reject the null hypothesis")

    x = np.linspace(
        stats.t.ppf(0.001, degrees_freedom),
        stats.t.ppf(0.999, degrees_freedom),
        500
    )
    y = stats.t.pdf(x, degrees_freedom)

    plt.figure(figsize=(9, 5))
    plt.plot(x, y)

    plt.fill_between(
        x[x <= lcv],
        stats.t.pdf(x[x <= lcv], degrees_freedom),
        alpha=0.3
    )

    plt.fill_between(
        x[x >= ucv],
        stats.t.pdf(x[x >= ucv], degrees_freedom),
        alpha=0.3
    )

    plt.axvline(lcv, linestyle="--", label="LCV")
    plt.axvline(ucv, linestyle="--", label="UCV")
    plt.axvline(test_statistic, label="Test statistic")

    plt.title("Welch t test with rejection regions")
    plt.xlabel("t value")
    plt.ylabel("Probability density")
    plt.legend()
    plt.show()

else:
    print("Not enough observations for the test")

## 15. Save the clean catalogue

The next notebook will use this file. This separation means we do not need to call the APIs every time we experiment with NLP.

In [ ]:
CLEAN_FILE = PROCESSED_DATA / "books_clean.csv"

books.to_csv(CLEAN_FILE, index=False)

print("Saved:", CLEAN_FILE)
print("Rows:", len(books))